# Headline → evidence workflow

This notebook walks one headline through the observatory: **headline → matched indicators → stored observation versions → screening rules → evidence card → 16:9 chart export**.

* It uses **live data** (`data/snapshot.duckdb`) when you have run `oco refresh`.
* If no live data exist it stops — unless you set `USE_DEMO = True`, which switches to the **SYNTHETIC** fixtures in `data_demo/` (built with `oco demo-build`). Synthetic output is watermarked and must never be used as evidence.
* All numbers are computed in Python from stored observations; the card's narrative is a template whose every number is validated.

In [ ]:
USE_DEMO = False   # set True ONLY to practise with synthetic fixtures

import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
from oco.settings import get_paths
from oco.storage.warehouse import open_snapshot

paths = get_paths(demo=USE_DEMO)
con = open_snapshot(paths)
if con is None:
    raise SystemExit("No snapshot. Run `oco refresh` (live) — or set USE_DEMO=True after `oco demo-build`.")
mode = (con.execute("SELECT value FROM meta WHERE key='mode'").fetchone() or [None])[0]
print("DATA MODE:", "SYNTHETIC DEMO — NOT LIVE" if mode == "DEMO_SYNTHETIC" else "live")
print(con.execute("SELECT source, COUNT(*) FROM series GROUP BY source").fetchall())

## 1. Pick a headline
The most recent headline that matched at least one topic (or paste your own with `oco news add`).

In [ ]:
cards = con.execute("""SELECT card_id, version, payload FROM evidence_cards
                        WHERE kind='headline' AND status='current' ORDER BY created_at DESC""").fetchall()
if not cards:
    raise SystemExit("No headline cards yet: refresh news (GDELT/RSS) or add one with `oco news add`, then `oco analyse`.")
card_id, version, payload = cards[0]
card = json.loads(payload)
print(card["headline"]["title"]); print(card["headline"]["url"])
print("published:", card["headline"]["published_at"], "| discovered:", card["headline"]["discovered_at"])
print("question:", card["question"])

## 2. The measurements behind the card
Each row comes from an immutable observation version (ids listed at the end).

In [ ]:
import pandas as pd
m = pd.DataFrame(card["calculations"]["measurements"])
cols = [c for c in ["name","unit","latest_value","latest_period","previous_value","previous_date","abs_change","pct_change","post_event","freshness"] if c in m]
m[cols]

## 3. Screening rules, relationship, alternatives

In [ ]:
for x in card["calculations"]["measurements"]:
    if x.get("anomaly"): print("-", x["series_id"], "→", x["anomaly"]["explanation"][:200])
print("\nRELATIONSHIP:", card["relationship"], "—", card["calculations"]["relationship_reason"])
if card["calculations"].get("temporal_note"): print(card["calculations"]["temporal_note"])
print("\nContrary/normal:", *card["contrary_observations"], sep="\n  ")
print("\nAlternatives:", *card["alternative_explanations"], sep="\n  ")
print("\nQuality dimensions:", card["quality"])

## 4. Provenance: the exact observation versions used

In [ ]:
ids = [i for i in card["input_version_ids"] if len(i) == 20]
prov = con.execute(f"""SELECT o.version_id, o.series_id, o.obs_start, o.value, o.revision_no, o.first_seen_at, o.raw_sha256, r.url
                       FROM observation_versions o LEFT JOIN raw_files r ON r.sha256 = o.raw_sha256
                       WHERE o.version_id IN ({','.join('?'*len(ids))})""", ids).df() if ids else pd.DataFrame()
prov

## 5. Export the card (16:9 charts + CSV + source note + card markdown)

In [ ]:
from oco.exports import export_card
out = export_card(con, card_id, paths.exports, demo=(mode == "DEMO_SYNTHETIC"))
print(out["dir"])
from IPython.display import Image, Markdown, display
first = next(iter(out["charts"].values()), None)
if first: display(Image(first["png"], width=900))
display(Markdown(Path(out["card"]).read_text()[:4000]))